# PatchTST vs DLinear on Electricity — Colab runner

This notebook reproduces the **Electricity** rows of Table 3 from Nie et al., *A Time Series is Worth 64 Words* (ICLR 2023).

**Before running**: Runtime → Change runtime type → **A100 GPU** (Colab Pro). T4 also works but is ~3× slower.

Total wall-clock on A100: ~6–8 hours for all 8 experiments. You can run a single experiment in 30–90 minutes.

## 1. Clone repo and install deps

In [ ]:
# Replace REPO_URL with your group's GitHub URL after pushing.
REPO_URL = 'https://github.com/<your-org>/<your-repo>.git'

import os
if not os.path.isdir('/content/Project'):
    !git clone $REPO_URL /content/Project
%cd /content/Project
!pip install -q -r code/requirements.txt

## 2. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 3. Download Electricity dataset

Pulls the pre-processed CSV from the Autoformer release bundle (same file the PatchTST authors used).

In [ ]:
!mkdir -p data/electricity
# gdown is bundled in the default Colab image; if not, install it.
!pip install -q gdown
# File ID for electricity.csv from the Autoformer Google Drive bundle.
# If this ID stops working, download manually from:
#   https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy
!gdown --id 1rWXSx5ITS-BXuOkZsEEbpZpa3Xz4PUAb -O data/electricity/electricity.csv || echo 'Download failed — please upload electricity.csv to data/electricity/ manually.'
!ls -la data/electricity/

## 4. Sanity-check data loader

In [ ]:
import sys
sys.path.insert(0, 'code')
from data_provider import build_dataloaders
loaders, c_in = build_dataloaders('data/electricity/electricity.csv', seq_len=336, pred_len=96, batch_size=32, num_workers=2)
for split, ld in loaders.items():
    x, y = next(iter(ld))
    print(f'{split:5s}  x={tuple(x.shape)}  y={tuple(y.shape)}  steps={len(ld)}')
print('num_features =', c_in)

## 5. Run a single experiment (e.g. PatchTST T=96)

In [ ]:
!python code/main.py --config code/configs/electricity_patchtst_96.yaml

## 6. Run **all 8 experiments** (long: ~6–8 GPU-hours on A100)

In [ ]:
!bash code/scripts/run_all.sh

## 7. Build summary CSV + plots

In [ ]:
!python code/scripts/summarize.py
!python code/scripts/plot_history.py

In [ ]:
import pandas as pd
pd.read_csv('results/summary.csv')

## 8. (Optional) Persist results back to GitHub or Drive

Easiest way to save the JSONs and `summary.csv` off Colab is to mount Google Drive and copy them over.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/cs4782_patchtst_results